# Assignment 4: Word Embeddings + Emotion Recognition

Based on GoEmotions dataset and Word2Vec/GloVe requirements.

**Detected dataset columns:** id, text, example_very_unclear, admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, json

df = pd.read_csv('go_emotions_dataset.csv')
print(df.shape)
df.head()


In [ ]:

# Text Preprocessing

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'[^a-z\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:

# Word2Vec Training

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

sentences = [word_tokenize(clean_text(str(x)))
             for x in df.iloc[:,0].astype(str)]

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,
    epochs=10
)

w2v_model.save('word2vec_custom.model')


In [ ]:

# Similarity & Analogy Examples

try:
    print(w2v_model.wv.most_similar('happy'))
except:
    print('Word not present')


In [ ]:

# t-SNE Visualization

from sklearn.manifold import TSNE

words = list(w2v_model.wv.index_to_key[:300])
vectors = np.array([w2v_model.wv[w] for w in words])

tsne = TSNE(n_components=2, random_state=42)
coords = tsne.fit_transform(vectors)

plt.figure(figsize=(10,8))
plt.scatter(coords[:,0], coords[:,1])

for i,w in enumerate(words[:100]):
    plt.annotate(w,(coords[i,0],coords[i,1]))
plt.show()


In [ ]:

# Load GloVe Embeddings

def load_glove(filepath):
    embeddings = {}
    with open(filepath,'r',encoding='utf-8') as f:
        for line in f:
            values=line.split()
            embeddings[values[0]] = np.asarray(values[1:],dtype='float32')
    return embeddings

# glove = load_glove('glove.6B.100d.txt')


In [ ]:

# Emotion Distribution

label_col = df.columns[-1]

plt.figure(figsize=(12,5))
df[label_col].value_counts().head(30).plot(kind='bar')
plt.title('Emotion Distribution')
plt.show()


In [ ]:

# TF-IDF Baseline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df.iloc[:,0].astype(str)
y = df.iloc[:,-1]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42)

tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf,y_train)

pred = lr.predict(X_test_tfidf)

print(classification_report(y_test,pred))


In [ ]:

# Neural Network with GloVe (Template)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(len(set(y)), activation='softmax')
])

model.summary()



## Required Assignment Sections

1. Word2Vec hyperparameter experiments
   - vector_size = 50,100,200,300
   - window = 2,5,10
   - CBOW vs SkipGram

2. GloVe emotion classification
   - Dense Network
   - CNN
   - BiLSTM

3. Baseline comparison
   - TF-IDF + Logistic Regression
   - TF-IDF + Random Forest

4. Visualizations (15+)
   - t-SNE
   - Confusion Matrix
   - F1 per emotion
   - Training Curves
   - Emotion Similarity Matrix
   - Error Analysis

5. Technical Report
   - Word2Vec Analysis
   - Emotion Recognition
   - Embedding Comparison
   - Error Analysis
